This notebook is an adaptation of this [guide](https://github.com/polaris-hub/polaris-method-comparison/blob/main/ADME_example/ML_Regression_Comparison.ipynb) to our case.

> See SectionA.2 for guidance on multiple testing for a large number of method (_e.g._, models) comparisons (>10).

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import yaml
from pathlib import Path
from collections import defaultdict

import pandas as pd
import numpy as np
from tqdm import tqdm
from scipy import stats
from scipy.stats import levene
from statsmodels.stats.multitest import multipletests
from sklearn.metrics import mean_squared_error, roc_auc_score, accuracy_score
import matplotlib.pyplot as plt

In [ ]:
# Make plot directory if it doesn't exist
plot_dir = Path("./plots")
plot_dir.mkdir(parents=True, exist_ok=True)
print(f"Plots will be saved to: {plot_dir.resolve()}")

proj_dir = Path("/cephyr/users/ribes/Alvis/mimer/stefano/TACK")
predictions_dir = proj_dir / "predictions"
checkpoint_dir = proj_dir / "checkpoints"
# protac_stan_preds_dir = Path("/mimer/NOBACKUP/groups/naiss2023-6-290/nils/PROTAC-STAN/results_multitask_20260119_0248/predictions/")
protac_stan_preds_dir = Path("/mimer/NOBACKUP/groups/naiss2023-6-290/nils/PROTAC-STAN/results_multitask_20260414_1401/predictions/")

In [ ]:
def clean_method_name(method: str, data: str = None) -> str:
    """Cleans method names for better visualization."""
    final_method = ''
    if 'protac-stan' in method.lower():
        return 'PROTAC-STAN'
    if 'xgb' in method.lower():
        final_method = 'XGB'
    if 'mlp' in method.lower():
        final_method = 'MLP'

    final_method += '-QR' if '_qr' in method.lower() else ''
    final_method += '-MVE' if '_mve' in method.lower() else ''
    final_method += '-BIN' if '_bin' in method.lower() else ''
    final_method += '-DMAX' if '_dmax' in method.lower() else ''
    final_method += '-DC50' if '_dc50' in method.lower() else ''
    
    data_features = []
    
    data_info = method if data is None else data
    
    if 'fp512r16' in data_info.lower():
        data_features.append('FP')
    if '_desc' in data_info.lower():
        data_features.append('Mol-Desc')
    if '_poi_ord' in data_info.lower():
        data_features.append('POI-Ord')
    if '_lig_ord' in data_info.lower():
        data_features.append('E3-Ord')
    if '_assay_time' in data_info.lower():
        data_features.append('Time')
    if '_cell_ord' in data_info.lower():
        data_features.append('Cell-Ord')
    if '_cell_text' in data_info.lower():
        data_features.append('Cell-Text')
    if '_cell_pt' in data_info.lower():
        data_features.append('Cell-Emb')
    if '_cell_onehot' in data_info.lower():
        data_features.append('Cell-OneHot')
    if '_poi_pt' in data_info.lower():
        data_features.append('POI-Emb')
    if '_poi_onehot' in data_info.lower():
        data_features.append('POI-OneHot')
    if '_poi_vec' in data_info.lower():
        data_features.append('POI-Vec')
    if '_lig_pt' in data_info.lower():
        data_features.append('E3-Emb')
    if '_lig_onehot' in data_info.lower():
        data_features.append('E3-OneHot')
    if '_lig_vec' in data_info.lower():
        data_features.append('E3-Vec')
    if '_poi_emb' in data_info.lower():
        data_features.append('POI-ESM-S')
    if '_lig_emb' in data_info.lower():
        data_features.append('E3-ESM-S')
    if '_poi_pca44_lig_pca' in data_info.lower():
        data_features.append('POI/E3-ESM-S-PCA')

    # Remove extra spaces
    final_method += ' ' + ' '.join(sorted(data_features))
    return final_method

# Get all files in the predictions directory
prediction_files = list(predictions_dir.glob("*.csv"))
protac_stan_files = list(protac_stan_preds_dir.glob("*.csv"))

# Load all CSV files in the predictions directory via a for loop
method2hparameters = {}
method2file = defaultdict(list)
raw_methods = set()
results = []
for file in sorted(prediction_files):
    # Extract model name from filename
    model_name = file.stem.split("=")[1].split("-")[0]
    data_name = file.stem.split("=")[2].split("-")[0]
    raw_methods.add((model_name + ' ' + data_name, clean_method_name(model_name, data_name)))
    method2file[model_name + ' ' + data_name].append(file.stem)
    
    # Setup the filename of the hyperparameters file
    if 'xgb' in model_name.lower():
        filename = str(file.stem.replace("preds", "config"))
    elif 'mlp' in model_name.lower():
        filename = str(file.stem.replace("preds", "best_config"))
    else:
        filename = ''
    # Remove '-fold=XX' and '-split=YYYY' from the filename
    filename = '-'.join([part for part in filename.split('-') if not part.startswith('fold=') and not part.startswith('split=')])
    filename = filename  + ".yaml"
    method2hparameters[clean_method_name(model_name, data_name)] = checkpoint_dir / filename

for file in sorted(protac_stan_files):
    model_name = 'PROTAC-STAN'
    raw_methods.add((model_name, clean_method_name(model_name)))
    method2file[model_name].append(file.stem)

for raw_method, method in sorted(raw_methods, key=lambda x: x[1]):
    print(f"{method:20} -> {raw_method}")
    
# Check that all the raw_method are unique
raw_method_names = [rm for rm, m in raw_methods]
assert len(raw_method_names) == len(set(raw_method_names)), "Raw method names are not unique!"

methods = [m[1] for m in raw_methods]

for method in methods:
    if method not in [m for rm, m in raw_methods]:
        raise ValueError(f"Method {method} not found in raw methods!")

# Check that all selected methods have the same number of files
num_files = [len(method2file[rm]) for rm, m in raw_methods if m in methods]
if len(set(num_files)) != 1:
    raise ValueError("Not all selected methods have the same number of files!")

In [ ]:
# ---------------------------------------------------------------------------
# Isolate the 10 feature-set configurations from Tab. X (Ord → OneHot variant)
# Each frozenset encodes the feature tokens that appear after the model-task
# prefix in the clean method name.
# ---------------------------------------------------------------------------

table_configs = {
    1:  frozenset(['Cell-OneHot', 'E3-OneHot', 'POI-OneHot', 'Mol-Desc', 'FP',   'Time']),
    2:  frozenset(['Cell-OneHot', 'E3-OneHot', 'POI-OneHot',             'FP',   'Time']),
    3:  frozenset(['Cell-OneHot', 'E3-OneHot', 'POI-OneHot', 'Mol-Desc',         'Time']),
    4:  frozenset(['Cell-OneHot', 'E3-OneHot', 'POI-Vec',    'Mol-Desc',         'Time']),
    # 1:  frozenset(['Cell-Ord', 'E3-Ord', 'POI-Ord', 'Mol-Desc', 'FP',   'Time']),
    # 2:  frozenset(['Cell-Ord', 'E3-Ord', 'POI-Ord',             'FP',   'Time']),
    # 3:  frozenset(['Cell-Ord', 'E3-Ord', 'POI-Ord', 'Mol-Desc',         'Time']),
    # 4:  frozenset(['Cell-Ord', 'E3-Ord', 'POI-Vec',    'Mol-Desc',         'Time']),
    5:  frozenset(['Cell-Text',   'E3-ESM-S',  'POI-ESM-S',  'Mol-Desc', 'FP',   'Time', 'POI/E3-ESM-S-PCA']),
    6:  frozenset(['Cell-Text',   'E3-ESM-S',  'POI-ESM-S',  'Mol-Desc',         'Time', 'POI/E3-ESM-S-PCA']),
    7:  frozenset(['Cell-Text',   'E3-OneHot', 'POI-Vec',    'Mol-Desc', 'FP',   'Time']),
    8:  frozenset(['Cell-Text',   'E3-OneHot', 'POI-OneHot', 'Mol-Desc',         'Time']),
    9:  frozenset(['Cell-Text',   'E3-OneHot', 'POI-Vec',    'Mol-Desc',         'Time']),
    10: frozenset([               'E3-ESM-S',  'POI-ESM-S',  'Mol-Desc', 'FP'         ]),
}

all_config_feature_sets = set(table_configs.values())

def feature_set_from_clean_name(clean_name: str) -> frozenset:
    """Returns the feature tokens of a clean method name (everything after the model-task prefix)."""
    parts = clean_name.strip().split()
    return frozenset(parts[1:])  # parts[0] is e.g. 'MLP-BIN' or 'XGB-DC50'

# Filter raw_methods to only the 10 configs
selected_raw_methods = {
    (rm, m)
    for rm, m in raw_methods
    if feature_set_from_clean_name(m) in all_config_feature_sets
}

# Build a reverse map: feature_set → config index (for verification printout)
feature_set_to_config = {fs: idx for idx, fs in table_configs.items()}

print(f"\nSelected {len(selected_raw_methods)} methods across {len(table_configs)} configs:\n")
for raw_method, method in sorted(selected_raw_methods, key=lambda x: x[1]):
    cfg_idx = feature_set_to_config[feature_set_from_clean_name(method)]
    print(f"  Config {cfg_idx:2d} | {method:60s} <- {raw_method}")

# Sanity check: every config should be covered by at least one method
covered_configs = {feature_set_to_config[feature_set_from_clean_name(m)] for _, m in selected_raw_methods}
missing_configs = set(table_configs.keys()) - covered_configs
if missing_configs:
    raise ValueError(f"No methods found for configs: {missing_configs}")
print(f"\nAll {len(table_configs)} configs covered ✓")

# Update methods to the filtered selection
methods = sorted({m for _, m in selected_raw_methods})

In [ ]:
# Load all CSV files in the predictions directory via a for loop
results = []
for file in tqdm(prediction_files, desc="Loading prediction files"):
    # Extract model name from filename
    model_name = file.stem.split("=")[1].split("-")[0]
    data_name = file.stem.split("=")[2].split("-")[0]
    
    clean_method = clean_method_name(model_name, data_name)
    if clean_method not in methods:
        continue
    
    df = pd.read_csv(file)
    df['method'] = clean_method

    # Rename columns for consistency
    df = df.rename(columns={'group': 'split', 'value_type': 'task', 'confidence': 'prob'})
    # Rename task column values, from 'dmax' to 'Dmax' and 'dc50' to 'DC50'
    df['task'] = df['task'].str.replace('dmax', 'Dmax')
    df['task'] = df['task'].str.replace('dc50', 'DC50')
    df['task'] = df['task'].str.replace('binary_class', 'bin')
    df['task'] = df['task'].str.replace('heldout', 'bin')

    # For XGBoost, the 'pred' column refers to probabilities, so we need to
    # rename it and then threshold it at 0.5 to get binary predictions
    if 'bin' in df['task'].unique()[0]:
        if 'prob' not in df.columns:
            df['prob'] = df['pred'].copy()
            df['pred'] = (df['prob'] >= 0.5).astype(int)

    if 'PROTAC-STAN' in file.stem:
        df['set'] = 'test' if 'heldout' in file.stem else 'val'
    else:
        df['set'] = 'test' if 'test' in file.stem else 'val'
    results.append(df)

for file in protac_stan_files:
    model_name = 'PROTAC-STAN'
    clean_method = clean_method_name(model_name)
    if clean_method not in methods:
        continue
    
    df = pd.read_csv(file)
    df['method'] = clean_method

    # Rename columns for consistency
    df = df.rename(columns={'group': 'split', 'value_type': 'task', 'confidence': 'prob'})
    # Rename task column values, from 'dmax' to 'Dmax' and 'dc50' to 'DC50'
    df['task'] = df['task'].str.replace('dmax', 'Dmax')
    df['task'] = df['task'].str.replace('dc50', 'DC50')
    df['task'] = df['task'].str.replace('binary_class', 'bin')
    df['task'] = df['task'].str.replace('multitask', 'bin')
    df['task'] = df['task'].str.replace('heldout', 'bin')

    df['set'] = 'test' if 'heldout' in file.stem else 'val'
    results.append(df)

results_df = pd.concat(results, ignore_index=True)

# For get the maximum number of folds for any method
max_folds = results_df.groupby('method')['fold'].nunique().max()

# Build a mask to keep only (method, task) pairs with the required number of folds
mask = []
for (method, task), group in results_df.groupby(['method', 'task']):
    num_folds = group['fold'].nunique()
    required_folds = max_folds
    if num_folds < max_folds:
        print(f"WARNING: Method '{method}' for task {task} has only {num_folds} folds (expected {required_folds})")
        # display(group[['method', 'task', 'fold']])
        mask.extend(group.index.tolist())

# Remove only the problematic (method, task) pairs
results_df = results_df.drop(mask).reset_index(drop=True)

# Print all methods
print("\nMethods found in results:")
for method in sorted(results_df['method'].unique()):
    print(f"- {method}")

def dc50_to_pdc50(x):
    """Convert DC50 in nM to pDC50."""
    return -np.log10(x * 1e-9 + 1e-12)

# Convert all task 'DC50' to pDC50 values by taking -log10, they are in nM
for task_group, group in results_df.groupby('task'):
    if task_group == 'DC50':
        results_df.loc[group.index, 'target'] = group['target'].apply(dc50_to_pdc50)
        results_df.loc[group.index, 'pred'] = group['pred'].apply(dc50_to_pdc50)
        
        if 'pred_lower' in results_df:
            results_df.loc[group.index, 'pred_lower'] = group['pred_lower'].apply(dc50_to_pdc50)
            results_df.loc[group.index, 'pred_upper'] = group['pred_upper'].apply(dc50_to_pdc50)

In [ ]:
def perform_bh_analysis(pivot_df, maximize, alpha):
    """
    Tests if ANY method is significantly better than the candidate best.
    If so, updates the control and repeats.
    """
    means = pivot_df.mean()
    best_method = means.idxmax() if maximize else means.idxmin()
    
    # 1. Friedman Test
    stat, p_friedman = stats.friedmanchisquare(*[pivot_df[col] for col in pivot_df.columns])
    print(f"\tFriedman Test: χ²={stat:.4f}, p={p_friedman:.4e}")
    
    if p_friedman >= alpha:
        print(f"  → No significant differences detected (all methods equivalent)")
        return pivot_df.columns.tolist(), None
    
    # 2. MODIFIED: Check if best is truly best (two-stage testing)
    print(f"\tCandidate Best: {best_method} (Mean: {means[best_method]:.4f})")
    print(f"\tFile: {method2hparameters[best_method]}")
    
    # Stage 1: Test if anything beats the candidate best
    p_values_stage1 = []
    comparison_methods = []
    effect_sizes = []
    
    for method in pivot_df.columns:
        if method == best_method:
            continue
        
        # One-sided test: Is this method better than candidate best?
        if maximize:
            # Test: method > best_method
            stat, p = stats.wilcoxon(
                pivot_df[method], 
                pivot_df[best_method],
                alternative='greater'
            )
        else:
            # Test: method < best_method
            stat, p = stats.wilcoxon(
                pivot_df[method], 
                pivot_df[best_method],
                alternative='less'
            )
        
        p_values_stage1.append(p)
        comparison_methods.append(method)
        effect_sizes.append(means[method] - means[best_method])
    
    # Apply BH to stage 1
    reject_stage1, pvals_corrected_stage1, _, _ = multipletests(
        p_values_stage1, alpha=alpha, method='fdr_bh'
    )
    
    # If any method significantly beats the candidate best, RECONSIDER
    if any(reject_stage1):
        better_methods = [m for m, r in zip(comparison_methods, reject_stage1) if r]
        print(f"  ⚠ WARNING: Method(s) {better_methods} may be significantly better!")
        print(f"  → Consider using rank-based selection instead of mean-based.")
        
        # Select the method with best median rank as control instead
        median_ranks = pivot_df.rank(ascending=not maximize, axis=1).median()
        best_method = median_ranks.idxmin()
        print(f"  → Updated control to: {best_method} (based on median rank)")
    
    # Stage 2: Now test others against validated best
    print(f"\t[Final Analysis: Testing against {best_method}]")
    
    p_values_stage2 = []
    comparison_methods_stage2 = []
    
    for method in pivot_df.columns:
        if method == best_method:
            continue
        
        # Two-sided test: Is this method different from best?
        stat, p = stats.wilcoxon(
            pivot_df[best_method], 
            pivot_df[method],
            alternative='two-sided'
        )
        
        p_values_stage2.append(p)
        comparison_methods_stage2.append(method)
    
    # BH correction for stage 2
    reject_stage2, pvals_corrected_stage2, _, _ = multipletests(
        p_values_stage2, alpha=alpha, method='fdr_bh'
    )
    
    winning_group = [best_method]
    
    print("\t[Benjamini-Hochberg Results]")
    report_df = []
    for method, is_rejected, p_corr in zip(comparison_methods_stage2, reject_stage2, pvals_corrected_stage2):
        mean_diff = means[best_method] - means[method]
        if is_rejected:
            # print(f"  ✗ {method}: Significantly Worse (p_adj={p_corr:.4e}, Δ={mean_diff:+.4f})")
            pass
        else:
            # print(f"  ✓ {method}: Equivalent (p_adj={p_corr:.4f}, Δ={mean_diff:+.4f})")
            winning_group.append(method)
        report_df.append({
            'Method': method,
            'Mean': means[method],
            'Mean Difference': mean_diff,
            'p_adj': p_corr,
            'Significantly Worse': is_rejected,
            
        })
    
    report_df = pd.DataFrame(report_df)
    # print(report_df.to_markdown(index=False))
    
    return winning_group, {
        'best_method': best_method,
        'n_equivalent': len(winning_group) - 1,
        'friedman_p': p_friedman,
        'best_mean': means[best_method]
    }


def compare_methods_benjamini_hochberg(df, subset='test', alpha=0.05):
    """
    Compares model configurations using Friedman test + Wilcoxon Signed-Rank test
    with Benjamini-Hochberg (FDR) correction for multiple comparisons.
    
    Args:
        df (pd.DataFrame): Columns [task, prob, pred, target, fold, method, set]
        subset (str): The value in 'set' column to filter by (e.g., 'test')
        alpha (float): FDR threshold (default 0.05)
        
    Returns:
        pd.DataFrame: Summary of the best configuration(s) per task.
    """
    
    # 1. Filter Data
    df_eval = df[df['set'] == subset].copy()
    if df_eval.empty:
        raise ValueError(f"No data found for set='{subset}'")

    results_summary = []
    
    # 2. Iterate per Task
    for task in sorted(df_eval['task'].unique()):
        print(f"\n{'='*30}\nAnalyzing Task: {task}\n{'='*30}")
        task_df = df_eval[df_eval['task'] == task]
        
        # 3. Detect Metric
        unique_targets = task_df['target'].unique()
        is_classification = (len(unique_targets) <= 2) and (set(unique_targets).issubset({0, 1}))
        
        # 4. Calculate Metrics Per Fold
        # Structure: Index=Fold, Columns=Method, Values=Score
        fold_metrics = []
        grouped = task_df.groupby(['method', 'fold'])
        
        for (method, fold), group in grouped:
            y_true = group['target']
            y_pred = group['pred']
            
            if is_classification:
                y_prob = group['prob']

                # AUC preferred; fallback to Accuracy if single-class fold
                if len(y_true.unique()) > 1:
                    score = roc_auc_score(y_true, y_prob)
                    metric_name = 'ROC-AUC'
                    maximize = True
                else:
                    score = accuracy_score(y_true, y_pred)
                    metric_name = 'Accuracy'
                    maximize = True
            else:
                score = np.sqrt(mean_squared_error(y_true, y_pred))
                metric_name = 'RMSE'
                maximize = False
            
            fold_metrics.append({'method': method, 'fold': fold, 'score': score})
            
        pivot_df = pd.DataFrame(fold_metrics).pivot(index='fold', columns='method', values='score')
        
        # Drop methods with missing folds to ensure paired testing works
        if pivot_df.isnull().values.any():
            print("  ! Warning: Dropping methods with missing folds.")
            pivot_df = pivot_df.dropna(axis=1)

        # 5. Statistical Analysis (Friedman + BH)
        best_configs, comparison_df = perform_bh_analysis(pivot_df, maximize, alpha)
        
        # Log Summary
        for conf in best_configs:
            results_summary.append({
                'Task': task,
                'Metric': metric_name,
                'Best Configuration': conf,
                'Mean Score': pivot_df[conf].mean(),
                'Significance': 'Best or Equivalent (FDR controlled)'
            })

    return pd.DataFrame(results_summary)


report_df = []

for dset in ['val']: # ['val', 'test']:
    for model in ['XGB', 'MLP']:
        print('\n' + '=' * 80)
        print(f"Analyzing model: {model} on dataset: {dset}")
        print('=' * 80)
        df = results_df[results_df['method'].str.contains(model)]
        if df['method'].nunique() < 2:
            print(f"Not enough configurations for model {model}, skipping...")
            continue
        winners_df = compare_methods_benjamini_hochberg(df, subset=dset, alpha=0.05)
        # print(f"\nFinal Recommended Configurations for {model}:\n")
        # print(winners_df.round(3).to_markdown(index=False))
        # print()
        winners_df['dset'] = dset
        report_df.append(winners_df)

report_df = pd.concat(report_df, ignore_index=True)

In [ ]:
def edit_config(s):
    # s = s.replace('-BIN', '')
    # s = s.replace(' Time', '')
    # s = s.replace('POI-ESM-S POI/E3-ESM-S-PCA', 'E3/POI-ESM-S (PCA)')
    return s

tmp = report_df.copy()
tmp['Best Configuration'] = tmp['Best Configuration'].apply(edit_config)

# Group by dset then print the final report ordered by Task and Method
for dset, group in tmp.groupby('dset'):
    print(f"\nFinal Report for dataset: {dset}\n")
    ordered_group = group.sort_values(by=['Task', 'Best Configuration'])
    print(ordered_group.round(3).to_markdown(index=False))
print("")

for dset, group in tmp.groupby('dset'):
    print(f"List of best configurations for dataset: {dset}\n")
    for task, task_group in group.groupby('Task'):
        print(f"  Task: {task}")
        for conf in task_group['Best Configuration']:
            print(f"    - {conf}")
    print("")

In [ ]:
best_methods = report_df['Best Configuration'].unique().tolist()
# For each hparam file, load it and print the hyperparameters for the best configuration
for method in best_methods:
    if method in report_df['Best Configuration'].values:
        hparams_file = method2hparameters[method]
        print(f"\nHyperparameters for {method}:")
        with open(checkpoint_dir / hparams_file, 'r') as f:
            hparams = yaml.safe_load(f)
            print(yaml.dump(hparams, default_flow_style=False))

In [ ]:
from notebooks.best_equivalent_models import build_full_report

report = build_full_report(
    results_df,
    model_types=["XGB", "MLP"],
    subsets=("val", "test"),
    correction="holm",          # FWER — recommended for reporting
    aggregate_repeats_k=5,      # for 5x5 CV; None to keep all 25 folds
)

In [ ]:
report

In [ ]:
results_df['task'].unique()

In [ ]:
from notebooks.best_equivalent_models import find_equivalent_best_set, build_full_report

# Single slice, verbose result object
res = find_equivalent_best_set(
    results_df,
    task="bin",
    model_type="XGB",
    subset="val",
    alpha=0.05,
    correction="holm",
    aggregate_repeats_k=5,   # 5x5 CV → collapse to 5 per-repeat means
)
print(res)
print()
print(res.per_method_table.to_markdown(index=False))
print()
res = find_equivalent_best_set(
    results_df,
    task="bin",
    model_type="MLP",
    subset="val",
    alpha=0.05,
    correction="holm",
    aggregate_repeats_k=5,   # 5x5 CV → collapse to 5 per-repeat means
)
print(res)
print()
print(res.per_method_table.to_markdown(index=False))
print()

In [ ]:
for task in ['val']: #['val', 'test']:
    for model_type in ['XGB', 'MLP']:
        print(f"\n{'='*40}\nTask: {task} | Model Type: {model_type}\n{'='*40}")
        task_model_report = report[(report['subset'] == task) & (report['model_type'] == model_type)]
        print(task_model_report.round(3).to_markdown(index=False))